In [7]:
#TODO make sure this renders in the github repo

# (GQA) Grouped Query Attention

![GQA Diagram](../showcase_images/GQA.png)

**From Llama 3 paper:** 
- "We use grouped query attention ([GQA; Ainslie et al. (2023)](https://arxiv.org/abs/2305.13245)) with $\mathbf{8}$ **key-value heads** to improve inference speed and to reduce the size of **key-value caches** during decoding."
- "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."


**From GQA paper:**

![GQA vs others Diagram](../showcase_images/GQA_vs.png)

**KV Cache:** The Keys and Values are stored during inference, this significantly speeds up the inference time, by reducing the GPU memory usage.

In [8]:
import EasyJupyter
import torch
import torch.nn as nn
import torch.nn.functional as F
from model.RoPE import apply_rotary_pos_emb
from typing import Optional

In [9]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        (GQA) group query attention.
        """
        super().__init__()
        self.cfg = cfg

        self.attn_heads = cfg.attn_heads
        self.num_kv_heads = cfg.num_kv_heads

        # Determine how many Query heads share a single KV head, e.g., 32/8 = 4.
        self.num_groups = self.attn_heads // self.num_kv_heads
        self.head_dim = cfg.d_model // self.attn_heads

        # Q Projection
        self.wq = nn.Linear(cfg.d_model, self.attn_heads * self.head_dim, bias=False)

        # K and V Projections
        self.wk = nn.Linear(cfg.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.num_kv_heads * self.head_dim, bias=False)

        # Final output projection back to d_model
        self.w_out = nn.Linear(self.attn_heads * self.head_dim, cfg.d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        combined_mask: torch.Tensor = None,
        kv_cache: Optional[tuple] = None,
    ) -> tuple[torch.Tensor, Optional[tuple]]:
        """
        Args:
            x: The input sequence.
            freqs_cis: The precomputed frequencies for RoPE.
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            kv_cache: Inference only, Tuple of (key, value) previous cache, that we will use and update append new data to it cache.
        
        Returns:
            A tuple of (attn_output, updated_kv_cache)
        """
        batch_size, context_len, _ = x.shape

        # === Linear Projection ===
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        # Reshape into discrete heads
        q = q.view(batch_size, context_len, self.attn_heads, self.head_dim)

        # K & V have different shapes than Q
        k = k.view(batch_size, context_len, self.num_kv_heads, self.head_dim)
        v = v.view(batch_size, context_len, self.num_kv_heads, self.head_dim)

        # === Apply RoPE ===
        q, k = apply_rotary_pos_emb(q, k, freqs_cis)

        # Transpose dimension for attention math
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # === KV cache ===
        if kv_cache is not None:
            past_k , past_v = kv_cache
            # Concatenate along the context_len dimension (dim=2)
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        # Update the K and V to be stored for the next generation step.
        updated_kv_cache = (k, v)

        # === Broadcast  K and V to match Q shape ===
        # K shape goes from (batch, 8, context_len, head_dim) -> (batch, 32, context_len, head_dim)
        k = torch.repeat_interleave(k, repeats=self.num_groups, dim=1)
        v = torch.repeat_interleave(v, repeats=self.num_groups, dim=1)

        # === Scaled Dot Product Attention
        # During Pre-Training we use the combined_mask that contains the causal_mask and the doc_mask.
        is_causal = True if (combined_mask is None and kv_cache is None) else False

        # During inference (when kv_cache is not None), is_causal=False
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=combined_mask, is_causal=is_causal)

        # Recombine heads and apply final projection
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.cfg.d_model)

        return self.w_out(out), updated_kv_cache

In [11]:
# @i-c
# Test 
from llama_configs import Llama3_scaled_down
from model.RoPE import precompute_freqs_cis

cfg = Llama3_scaled_down()
d_model = cfg.d_model
context_len = 10
batch_size = 2
head_dim = cfg.d_model // cfg.attn_heads

GQA = GroupedQueryAttention(cfg).to(cfg.device)

dummy_input = torch.randn(batch_size, context_len, d_model, device=cfg.device)

# Pre-compute ROPE frequencies
max_generation_len = context_len + 5 # Pre-compute ROPE frequencies for larger than the initial context length
freqs_cis = precompute_freqs_cis(cfg, dim=head_dim, end=max_generation_len)
freqs_cis_pretrain = freqs_cis[:context_len] # Slice out just the frequencies for the initial prompt


# TEST PRE-TRAINING PASS (NO KV CACHE)
out_pretrain, kv_cache_new = GQA(dummy_input, freqs_cis_pretrain, combined_mask=None, kv_cache=None)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {out_pretrain.shape}")
print(f"Returned Cache Status: {type(kv_cache_new)} with K shape {kv_cache_new[0].shape}")

assert out_pretrain.shape == dummy_input.shape, "Output shape changed!"


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM
/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model
Input shape: torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])
Returned Cache Status: <class 'tuple'> with K shape torch.Size([2, 8, 10, 8])


In [12]:
# @i-c
# Test 2: Inference pass with KV cache
# Simulate generating a single token
x_single_token = torch.randn(batch_size, 1, cfg.d_model, device=cfg.device)

# We have to slice freqs_cis to provide only the rotation for the current sequence position
freqs_cis_single = freqs_cis[context_len : context_len + 1]

# Pass in the cache generated from Test 1
out_inference, kv_cache_updated = GQA(
    x_single_token, freqs_cis_single, combined_mask=None, kv_cache=kv_cache_new
)

print(f"Input shape: {x_single_token.shape}")
print(f"Output shape: {out_inference.shape}")
print(f"Updated KV cache shape: {kv_cache_updated[0].shape}")

assert out_inference.shape == x_single_token.shape, "Inference output shape mismatch!"
assert (
    kv_cache_updated[0].shape[2] == context_len + 1
), "Cache did not append correctly!"

Input shape: torch.Size([2, 1, 256])
Output shape: torch.Size([2, 1, 256])
Updated KV cache shape: torch.Size([2, 8, 11, 8])
